# 03 · ASVspoof 5 — LCNN e ablation (CPU)

Baseline leggera sui Mel-spettrogrammi del notebook 01, pensata per girare **senza GPU** (es. sul portatile).
LCNN con Max-Feature-Map (Lavrentyeva et al., Interspeech 2019) più tre componenti recenti, aggiunti uno alla volta:

| run | cosa aggiunge |
|---|---|
| `lcnn_asvspoof5` | LCNN, media sul tempo, cross-entropy |
| `lcnn_lstm_asvspoof5` | + BiLSTM e attention pooling (come LCNN-LSTM di ASVspoof 2021) |
| `lcnn_lstm_specaug_asvspoof5` | + SpecAugment in training (Park et al., 2019) |
| `lcnn_lstm_specaug_oc_asvspoof5` | + OC-Softmax (Zhang et al., IEEE SPL 2021) |

Stesso protocollo, stesse metriche e stesso formato dei risultati di XLSR-MamBo (notebook 02):
epoca scelta solo sul dev, eval valutato una volta, 3 seed. Ingresso: 128 frame Mel (≈ 4.1 s, la stessa
durata usata per MamBo). Con OC-Softmax il punteggio è α·cos: l'EER è confrontabile, actDCF e Cllr no
(il punteggio non è un log-likelihood ratio).

Servono solo le feature in `artifacts/features/asvspoof5/mel_sr16000_m128_fft1024_hop512/` e i TSV di
protocollo in `data/asvspoof5/` (non l'audio).

## 0 · Setup

In [ ]:
import json
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # il notebook gira da notebooks/
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis import config, report
from thesis.train import TrainConfig

print(config.describe())

## 1 · Configurazioni dell'ablation

In [ ]:
NAMES = ["lcnn_asvspoof5", "lcnn_lstm_asvspoof5", "lcnn_lstm_specaug_asvspoof5", "lcnn_lstm_specaug_oc_asvspoof5"]
CONFIGS = {n: REPO / "configs" / f"{n}.json" for n in NAMES}
SEEDS = [1, 2, 3]
LOG_FILE = report.run_dir("lcnn_ablation.log")  # un unico log per tutta la coda

cfgs = {n: TrainConfig.from_json(p) for n, p in CONFIGS.items()}
display(pd.DataFrame({n: {"device": c.device, "pooling": c.model.pooling, "specaug": c.model.specaug,
                          "loss": c.model.loss, "lr": c.lr, "batch": c.batch_size, "epoche max": c.epochs}
                      for n, c in cfgs.items()}).T)

feat_dir = config.ARTIFACTS_ROOT / "features" / "asvspoof5" / cfgs[NAMES[0]].mel_tag
n_npy = len(list(feat_dir.glob("*.npy"))) if feat_dir.exists() else 0
print(f"Spettrogrammi trovati in {feat_dir}: {n_npy} (attesi 25000)")

## 2 · Lancio in background

Le 4 config partono una dopo l'altra, ognuna con i 3 seed, in un processo separato che continua anche
chiudendo VS Code. Rilanciando, i seed completati vengono saltati e quello interrotto riprende
dall'ultima epoca. Il PC non deve andare in sospensione.

In [ ]:
LAUNCH = False  # metti True per avviare

if LAUNCH:
    attivi = [n for n in NAMES if report.processo_attivo(CONFIGS[n].name)]
    assert not attivi, f"training già attivo: {attivi}"
    pid = report.lancia([CONFIGS[n].relative_to(REPO) for n in NAMES], SEEDS, "lcnn_ablation.log")
    print("Avviato, PID", pid)

## 3 · Monitoraggio

Stato di tutte e 4 le run. Le config in coda dietro a quella attiva compaiono come "nessun training attivo":
è normale finché non tocca a loro. Con `AGGIORNA_MIN > 0` la cella si aggiorna da sola ogni minuto.

In [ ]:
AGGIORNA_MIN = 0  # es. 60 = aggiorna ogni minuto per un'ora

def stato_ablation():
    for n in NAMES:
        report.stato(n, CONFIGS[n].name, SEEDS, LOG_FILE)
        print()

report.ogni_minuto(stato_ablation, AGGIORNA_MIN)

## 4 · Curve di training (EER dev per epoca)

In [ ]:
fig, axes = plt.subplots(1, len(NAMES), figsize=(4 * len(NAMES), 3.2), sharey=True)
for ax, n in zip(axes, NAMES):
    for s in SEEDS:
        p = report.run_dir(n) / f"seed{s}" / "log.csv"
        if p.exists():
            log = pd.read_csv(p)
            ax.plot(log.epoca, 100 * log.dev_eer, marker=".", label=f"seed {s}")
    ax.set(title=n.replace("_asvspoof5", ""), xlabel="epoca")
axes[0].set_ylabel("EER dev (%)"); axes[0].legend()
plt.tight_layout()

## 5 · Risultati dell'ablation

Media ± deviazione standard sui seed completati (colonna `seed n`). Se i risultati di XLSR-MamBo
(notebook 02) sono presenti nello stesso `artifacts/`, vengono aggiunti per il confronto.

In [ ]:
runs = NAMES + (["mambo_asvspoof5"] if report.run_dir("mambo_asvspoof5").exists() else [])
per_seed = report.risultati(runs, SEEDS)
assert len(per_seed), "Nessun seed completato: guarda la sezione 3"
display(per_seed.round(4))
summary = report.riepilogo(per_seed)
display(summary)

In [ ]:
eval_eer = per_seed[per_seed.split == "eval"].groupby("run")["EER %"].agg(["mean", "std"]).reindex(runs).dropna()
ax = eval_eer["mean"].plot.barh(xerr=eval_eer["std"], figsize=(7, 0.6 * len(eval_eer) + 1), title="EER eval (%), media ± std sui seed")
ax.invert_yaxis()
plt.tight_layout()

### EER per attacco (eval): dove aiutano i componenti

In [ ]:
per_attack = pd.DataFrame({n.replace("_asvspoof5", ""): report.eer_per_gruppo(n, SEEDS, "eer_per_attacco")["media"]
                           for n in runs if (report.run_dir(n) / "seed1" / "metrics.json").exists()})
display(per_attack.round(2))
per_attack.plot.bar(figsize=(14, 4), title="EER per attacco — eval (%), media sui seed")
plt.tight_layout()